# Travel Assist Bot Demo Query

This notebook sends a trip-planning request to the FastAPI `POST /plan` endpoint **in-process** (no uvicorn, no external host) and renders a presentation-friendly result.

## How to run
1. Open this notebook from the project root
2. Select TravelBot (.venv)(Python 3.13.11) from Jupyter Kernel.
3. Run all cells in order
4. The notebook calls `/plan` using `fastapi.testclient.TestClient` directly against `app.orchestrator.api:app`


In [ ]:
import sys
import json
from pathlib import Path
from urllib.parse import quote_plus

from fastapi.testclient import TestClient
from IPython.display import Markdown, display

# Make project imports reliable from either repo root or notebooks/ working dir.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "app").exists() and (PROJECT_ROOT.parent / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.orchestrator.api import app

DEMO_QUERY = "3 days Kuala Lumpur budget 1000 food"
client = TestClient(app)

display(Markdown(
    f"**Execution mode:** In-process FastAPI TestClient  \
**Endpoint path:** `/plan`  \
**Demo query:** `{DEMO_QUERY}`"
))


In [ ]:
def manual_payload_from_query(query: str) -> dict:
    # Safe fallback payload compatible with app/orchestrator/api.py::TripRequest
    return {
        "origin": "Singapore",
        "destinations": ["kuala lumpur"],
        "budget": 1000,
        "duration_days": 3,
        "travelers": 1,
        "preferences": ["food"],
        "feedback": None,
        "session_memory": None,
        "include_state": False,
    }

def build_payload(query: str) -> tuple[dict, str]:
    """Try parse_trip_request first; fallback to a manual valid payload."""
    try:
        from app.common.request_parser import parse_trip_request

        parsed = parse_trip_request(query) or {}
        payload = {
            "origin": parsed.get("origin", "Singapore"),
            "destinations": parsed.get("destinations", []),
            "budget": parsed.get("budget"),
            "duration_days": parsed.get("duration_days", 4),
            "travelers": parsed.get("travelers", 1),
            "preferences": parsed.get("preferences", []),
            "feedback": None,
            "session_memory": None,
            "include_state": False,
        }

        # Minimal guardrails to ensure /plan compatibility
        if not payload["destinations"]:
            payload["destinations"] = ["kuala lumpur"]
        if payload["budget"] is None:
            payload["budget"] = 1000

        return payload, "parse_trip_request"
    except Exception as exc:
        return manual_payload_from_query(query), f"manual_fallback ({type(exc).__name__})"

payload, payload_mode = build_payload(DEMO_QUERY)
display(Markdown(
    f"**Payload mode:** `{payload_mode}`  \
**Payload JSON:**\n\n```json\n{json.dumps(payload, indent=2)}\n```"
))


In [ ]:
def _to_float(value):
    if isinstance(value, (int, float)):
        return float(value)
    if value is None:
        return None
    text = str(value).replace(",", "")
    digits = "".join(ch for ch in text if ch.isdigit() or ch == ".")
    if not digits:
        return None
    try:
        return float(digits)
    except Exception:
        return None

def _fmt_money(value):
    amount = _to_float(value)
    if amount is None:
        return "Not available"
    if amount.is_integer():
        return f"SGD {int(amount)}"
    return f"SGD {amount:.2f}"

def _link(label, url):
    if not url:
        return ""
    return f"[{label}]({url})"

def _search_link(text, city=""):
    q = f"{text} {city}".strip()
    return f"https://www.google.com/search?q={quote_plus(q)}"

def render_plan_markdown(result: dict, fallback_payload: dict) -> str:
    request_data = result.get("request", fallback_payload) or fallback_payload
    executor = result.get("executor", {}) or {}
    reviewer = result.get("reviewer", {}) or {}
    ranking = result.get("ranking", {}) or {}
    selected_variant = result.get("selected_variant", {}) or {}

    destinations = request_data.get("display_destinations") or request_data.get("destinations") or []
    destination_text = ", ".join(destinations) if destinations else "Not detected"
    duration = request_data.get("duration_days", "-")

    selected_plan = (
        selected_variant.get("variant_label")
        or executor.get("variant_label")
        or "Recommended Plan"
    )

    score_pct = ranking.get("score_pct")
    if isinstance(score_pct, (int, float)):
        quality = f"{score_pct:.0f}%"
    else:
        quality = "Not scored"

    cost_breakdown = executor.get("cost_breakdown", {}) or {}
    estimated_cost = (
        ranking.get("estimated_total")
        or reviewer.get("estimated_total")
        or cost_breakdown.get("total")
    )
    budget = request_data.get("budget")

    travel = executor.get("travel_details", {}) or {}
    flight = travel.get("flight", {}) or {}
    hotel = travel.get("hotel", {}) or {}
    transport = travel.get("transport", {}) or {}

    daily_itinerary = executor.get("daily_itinerary", []) or []
    restaurants = executor.get("restaurants", []) or []
    variants = result.get("variants", []) or []

    lines = []
    lines.append("## Trip Plan Summary")
    lines.append("")
    lines.append(f"- **Destination:** {destination_text}")
    lines.append(f"- **Duration:** {duration} days")
    lines.append(f"- **Selected Plan:** {selected_plan}")
    lines.append(f"- **Score / Quality:** {quality}")
    lines.append(f"- **Estimated Cost:** {_fmt_money(estimated_cost)}")
    lines.append(f"- **Budget:** {_fmt_money(budget)}")

    lines.append("")
    lines.append("## Cost Breakdown")
    lines.append("")
    lines.append(f"- Flight: {cost_breakdown.get('flight', 'Not available')}")
    lines.append(f"- Hotel: {cost_breakdown.get('hotel', 'Not available')}")
    lines.append(f"- Activities: {cost_breakdown.get('activities', 'Not available')}")
    lines.append(f"- Transport: {cost_breakdown.get('local_transport', 'Not available')}")
    lines.append(f"- Food: {cost_breakdown.get('food', 'Not available')}")
    lines.append(f"- Total: {cost_breakdown.get('total', _fmt_money(estimated_cost))}")

    lines.append("")
    lines.append("## Travel Details")
    lines.append("")
    lines.append(f"- Flight: {flight.get('suggestion', 'Not available')}")
    lines.append(f"  - Price: {flight.get('estimated_price', 'Not available')}")
    if flight.get("search_link"):
        lines.append(f"  - Search: {_link('Open search', flight.get('search_link'))}")

    lines.append(f"- Hotel: {hotel.get('name', 'Not available')}")
    lines.append(f"  - Price: {hotel.get('estimated_price', 'Not available')}")
    lines.append(f"  - Location: {hotel.get('location_note', 'Not available')}")
    if hotel.get("booking_link"):
        lines.append(f"  - Booking: {_link('Open booking search', hotel.get('booking_link'))}")

    lines.append(f"- Local Transport: {transport.get('mode', 'Not available')}")
    lines.append(f"  - Cost: {transport.get('estimated_cost', 'Not available')}")
    lines.append(f"  - Notes: {transport.get('notes', 'Not available')}")

    lines.append("")
    lines.append("## Day-by-Day Itinerary")
    lines.append("")
    if daily_itinerary:
        for item in daily_itinerary:
            if not isinstance(item, dict):
                lines.append(f"- {item}")
                continue

            day = item.get("day", "-")
            title = item.get("title", "Travel plan")
            city = item.get("city") or destination_text
            lines.append(f"- **Day {day}:** {_link(title, _search_link(title, city))}")

            details = item.get("details")
            if details:
                lines.append(f"  - Details: {details}")

            for part in ["morning", "lunch", "afternoon", "evening"]:
                text = str(item.get(part, "")).strip()
                if text:
                    label = part.capitalize()
                    lines.append(f"  - {label}: {_link(text, _search_link(text, city))}")
    else:
        lines.append("- No itinerary details returned.")

    lines.append("")
    lines.append("## Food and Restaurants")
    lines.append("")
    if restaurants:
        for r in restaurants:
            if isinstance(r, dict):
                name = r.get("name", "Restaurant")
                distance = r.get("distance_note", "Distance not provided")
                link = r.get("search_link") or _search_link(name, destination_text)
                lines.append(f"- {_link(name, link)} ({distance})")
            else:
                lines.append(f"- {r}")
    else:
        lines.append("- No restaurant recommendations returned.")

    lines.append("")
    lines.append("## Other Options")
    lines.append("")

    current_selected_key = str(selected_variant.get("variant_key", "")).strip().lower()
    alternatives = []
    for v in variants:
        if not isinstance(v, dict):
            continue
        if str(v.get("variant_key", "")).strip().lower() == current_selected_key:
            continue
        label = v.get("variant_label", "Alternative")
        vr = v.get("ranking", {}) or {}
        pct = vr.get("score_pct")
        total = vr.get("estimated_total")
        pct_text = f"{pct:.0f}%" if isinstance(pct, (int, float)) else "Not scored"
        alternatives.append(f"- {label}: {pct_text}, {_fmt_money(total)}")

    if alternatives:
        lines.extend(alternatives[:3])
    else:
        lines.append("- No alternative variants returned.")

    return "\n".join(lines)


In [ ]:
result = None
response_status_code = None
error_message = None

try:
    response = client.post("/plan", json=payload)
    response_status_code = response.status_code

    if response.status_code != 200:
        error_message = (
            f"In-process /plan returned non-200 response: {response.status_code}. "
            f"Body: {response.text[:500]}"
        )
    else:
        try:
            result = response.json()
        except Exception as exc:
            error_message = f"Response was not valid JSON: {type(exc).__name__}: {exc}"
except Exception as exc:
    error_message = f"In-process request failed: {type(exc).__name__}: {exc}"

if error_message:
    display(Markdown(f"## Execution Error\n\n{error_message}"))
else:
    missing_top_level = [
        k for k in ["request", "executor", "reviewer", "ranking", "selected_variant", "variants"]
        if k not in result
    ]

    if missing_top_level:
        display(Markdown(
            "## Response Warning\n\n"
            f"Missing expected fields: `{', '.join(missing_top_level)}`. "
            "Rendering will continue with fallbacks."
        ))

    display(Markdown(render_plan_markdown(result, payload)))


In [ ]:
if result is not None:
    display(Markdown("## Raw Response (trimmed)"))
    print(json.dumps(result, indent=2)[:5000])


In [ ]:
# Final validation checks for demo readiness
assert response_status_code == 200, f"Expected status 200, got {response_status_code}"
assert isinstance(result, dict), "Expected JSON object result"
assert "executor" in result, "Missing 'executor' in response"
assert isinstance(result["executor"], dict), "'executor' must be an object"
assert "daily_itinerary" in result["executor"], "Missing 'daily_itinerary' in executor"
assert isinstance(result["executor"]["daily_itinerary"], list), "'daily_itinerary' must be a list"
assert len(result["executor"]["daily_itinerary"]) >= 1, "Expected at least one itinerary day"

display(Markdown("## Validation\n\nAll required validation checks passed."))
